##NOTE: I need to get pictures of houses alongside their data such as:
number of rooms, square footage, prices, maybe year listed too. 

# Images I will take of each home: 
    Take images of front of house, backyard, kitchen, living room, dining room, bathroom. and master bedroom. 
    
I also need some way that the cluade AI can make an inference off of the images provided. (Maybe on modern kitchen or not, is there natural lighting, hard wood floor, does it look luxurious, the hosuing style (modern or buglaow))

Worst case scenario I can not do the iamge analysis part and only do the housing prices prediction. (This could also use house prices from other houses in neightborhoods to get average house price per neightborhhd.)



##Features I could query the LLM to make inferences on the house price:
    Ovreall interior condition, 
    Kitchen Quality (stone countertops, modern applicance), 
    bathroom quality (tile quality, vanity type, shower/tub style),
    natural lighting level (low / medium / high),
    ceiling height, 
    exterior material (brick, siding, stucco, stone)
    garage presence and size (num stalls)
    fencing quality
    lot slope,
    window size 

    house style? (traditional/classic, modern, common residential, luxury, a regional style)

Note:mlater I could also try to factor in neighborhoods as well. Take groups of housing listings from various neightborhoods and analyze their data alongside other factors related to the neighborhood( Walkabilityu, avg income, school ratings, crime index)

In [ ]:
CREATE OR REPLACE STAGE house_data_stage
    DIRECTORY = (ENABLE = TRUE)
    FILE_FORMAT = (TYPE = 'CSV'); 

In [ ]:
CREATE OR REPLACE STAGE house_images 
	DIRECTORY = ( ENABLE = true ) 
	ENCRYPTION = ( TYPE = 'SNOWFLAKE_SSE' );

In [ ]:
CREATE OR REPLACE TABLE house_data (
    house_id INT,
    address STRING, 
    price INTEGER,  
    bedrooms INTEGER, 
    bathrooms INTEGER, 
    sqft INTEGER, 
    url_link STRING,
    image_path STRING
);

In [ ]:
SHOW CURRENT_ACCOUNT()

# Prior to runng the below 2 cells, I added a folder of house images, and a XLSX file to snowflake using SnowCLI

To do so, I used the folowing commands: 
snow stage copy     "C:\Users\<Username>\house_price_predictor\my_custom_dataset\images" "@house_images_stage" --recursive
snow stage copy "C:\Users\1999d\OneDrive\Desktop\house_price_predictor\my_custom_dataset\Housing_Data.csv" "@house_data_stage" 

In [ ]:
LIST @house_data_stage 

In [ ]:
REMOVE @house_images_stage

With the data uploaded to snowflake stages, I can now load it into the table.

NOTE: May need to remove commas from the column titles in the excel file.

In [ ]:
SELECT * 
FROM image_paths_raw;

I have to rerun the below cell each snowflake session since it is a temp table, thus when the session closes I lose all data that existed inside that table. I THINK CHECK LATER

In [ ]:
/*
The below 2 querries load the csv data into a temp table 
that will later be used to create the final table with both csv data and images.
*/
CREATE OR REPLACE TEMP TABLE excel_data_raw (
    house_id INT,
    address STRING,
    price INT,
    bedrooms INT,
    bathrooms INT,
    sqft INT,
    url_link STRING
);

In [ ]:
COPY INTO excel_data_raw
FROM @house_data_stage/Housing_Data.csv
FILE_FORMAT = (
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
);

In [ ]:
SELECT * 
FROM excel_data_raw;

Following loading the csv data into the temp table, I now needed to upload the images the house_data table.

In [ ]:
/*
This table will be used to hold the paths of the images saved in th esnowflake stage.
*/
CREATE OR REPLACE TEMP TABLE image_paths_raw (
    house_id INT,
    image_path STRING UNIQUE
);


In [ ]:
from snowflake.snowpark.functions import col
from snowflake.snowpark.context import get_active_session
import os

# Initialize the snowflake session
sesh = get_active_session()

# List files in the stage
files = sesh.sql("LIST @house_images").collect()

rows = []

for f in files:
    # f.name looks like: '1_1.jpg'
    filename = f.name.replace('house_images/', '')  # remove stage prefix if present

    # Skip directories or weird entries
    if not filename.lower().endswith((".jpg")):
        continue

    # Expect format: houseid_imagenum.png
    if "_" not in filename:
        continue

    # Split into house_id and the rest
    house_id_str, image_num = filename.split("_", 1)

    # Remove extension from house_id part if needed
    house_id = int(house_id_str)

    rows.append((house_id, filename))


# Upload house ID and filenames to SQL table
if rows:
    # Create DataFrame with eahc entry a tuple of house ID and filename 
    df = sesh.create_dataframe(rows, schema=["house_id", "filename"])

    # Save each tuple as a row in the SQL table
    df.write.save_as_table("image_paths_raw", mode="append")
    print(f"Inserted {len(rows)} rows into image_paths_raw.")
else:
    print("No valid image files found — check LIST output.")
    
# print(f"\n\n Rows:\n\n, {rows}")

In [ ]:
SELECT *
FROM image_paths_raw

In [ ]:
/*
Now I can upload the table house_data with the data stored inside of both temp tables above
*/

INSERT INTO house_data (
    house_id,
    address,
    price,
    bedrooms,
    bathrooms,
    sqft,
    url_link,
    image_path
)

SELECT
    i.house_id,
    e.address,
    e.price,
    e.bedrooms,
    e.bathrooms,
    e.sqft,
    e.url_link,
    i.image_path
FROM image_paths_raw i
LEFT JOIN excel_data_raw e
    ON i.house_id = e.house_id;

In [ ]:
SELECT * 
FROM house_data

In [ ]:
LIST @SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES;


NOTE: I kept receiving an error while trying to query the LLM as it was saying snowflake cant decrypt or work with client side encryption. So I had to create a new stage in the GUI, and set the stage encryption type used to server side. This was for the below cell.


In [ ]:
# With all of the data uploaded into the SQL talbe, 
# I can now query the LLM to perform an inference on the images.

In [ ]:
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA()

In [ ]:
# Load all image paths from your SQL table
rows = sesh.table("image_paths_raw").collect()

print(rows)

In [ ]:
"""
In this cell I ask a LLM model (Claude-4.5) to make an
inference based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the style of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Modern
- Ranch-style
- Raised ranch-style
- Two-story traditional
- Bungalow
- Prairie Suburban
- Farmhouse / Modern Farmhouse
- Chalet / Mountain-influenced
- Four Square
- Skinny homes 
- Mediterranean

If you are unable to determine the type from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
house_predictions = {}

for house_id in range(1,21):

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_1.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_1.jpg': {e}")
        continue

    # Initialize list if needed
    if house_id not in house_predictions:
        house_predictions[house_id] = []

    house_predictions[house_id].append(prediction)

    print(house_id, prediction)
    
print("\n\n\n")
print(house_predictions)

In [ ]:
# This cell cleans up the values in the 
# house_predictions dictionary created in the above cell.
cleaned = {}

for key, value in house_predictions.items():
    cleaned[key] = value.pop().strip('"')
  
house_predictions = cleaned

print(house_predictions)

# NOTE: Below cell s has images that are not perfectly formatted. I will need to go back and try and fix this. (White bar at top and bottom of images --> is there spme wayto fit the box surrounding the images to the size of the image? Or can I at least change that box background color from white to somehting that blends in better????)

In [ ]:
# This cell displays the images analyzed in gallery format 
# using Streamlit:

# streamlit is used to build the image gallery that the bear 
# images are being displayed in.
import streamlit 
import os
import base64


# Display a title using streamlit
streamlit.title("Houses Being Analyzed:")
streamlit.markdown("### These are only the front exteriors of the homes.")
streamlit.markdown("### The exteriors will be displayed alongside the interiors later.")


cols = streamlit.columns(4)
idx =1

while True:

    try:
        filename = f"{idx}_1.jpg"

        # Load image bytes from stage
        file_bytes = sesh.file.get_stream(f"@HOUSE_IMAGES/{filename}").read()
        encoded = base64.b64encode(file_bytes).decode()
    
        # Choose column (0–3)
        col = cols[(idx - 1) % 4]
    
        # Below uses streamlit to create a grid of 4 columns
        # # where each container in the grid can hold images
        with col:
            streamlit.markdown(
                f"""
                <div style='
                    display: flex;
                    justify-content: center;
                    align-items: center;
                    height: 250px;
                    border: 1px solid grey;
                    padding: 10px;
                    border-radius: 8px;
                    margin-bottom: 10px;
                '>
                    <img src="data:image/jpeg;base64,{encoded}"
                         style="max-width: 100%; max-height: 100%; object-fit: contain;">
                </div>
                <p style='text-align:center; color:gray;'>House {idx}</p>
                """,
                unsafe_allow_html=True
            )

            idx+=1
        
    except Exception:
        print(f"Reached end of houses.")
        break    

In [ ]:
"""
I can also ask LLM to rate the interior of the homes based on 

    Luxury Level (Basic, Mid, High End)
        hardwoord vs laminate, custom cabinetry, high end fixtures, tile quality

    Lot size (Small , medium, large) 
        Tight suburban lot, medium fenced yard, arecage / rural     
"""

In [ ]:
"""
In this cell I ask a LLM model (Claude-3.5) to make an
inference on the luxury level based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the level of luxury of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Basic:
    Flooring: Laminate, Vinyl plank, basic carpet, repeating patterns 
    Cabinetry: Stock cabinets, standard hardware
    Tile & Bathrooms: Ceramic tile, simple grid layout
    Appliances: Standard white/black/stainless
    Countertops: Laminate, basic quartz
    Windows: Standard vinyl, basic blinds

- Mid:
    Flooring: Engineered hardwood, high‑quality laminate, wider planks
    Cabinetry: Semi‑custom, plywood boxes, soft‑close hinges, upgraded hardware
    Tile & Bathrooms: Porcelain tile, larger formats
    Appliances: Mid‑tier stainless
    Countertops: Mid‑grade quartz, granite
    Windows: Upgraded vinyl

- High End: 
    Flooring: Solid hardwood, exotic woods, herringbone/chevron patterns 
    Cabinetry: Fully custom cabinetry, solid wood
    Tile & Bathrooms: Natural stoneslab walls, 
    Appliances: Luxury brands
    Countertops: High‑end quartz, marble, quartzite, waterfall edges
    Windows: Large architectural windows


If you are unable to determine the luxury level from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
luxury_predictions = {}
house_id = 1 
image_id = 1

while house_id < 22:

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_{image_id}.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_{image_id}.jpg': {e}")
        image_id = 1
        house_id +=1
        continue

    prediction_id = str(house_id) + '_' + str(image_id)
    
    # Initialize list if needed
    if prediction_id not in luxury_predictions:
        luxury_predictions[prediction_id] = []

    luxury_predictions[prediction_id].append(prediction)

    image_id += 1

    print(prediction_id, prediction)
    
print("\n\n\n")
print(luxury_predictions)

In [ ]:
"""
In this cell I ask a LLM model (Claude-3.5) to make an
inference on the lot size based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the lot size of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Small: 
    Tight suburban lot

- Medium: 
    Medium fenced yard
    Wide garage
    full front yard
    driveway fits 2-4 cars
    lot width around 45-70ft

- Large: 
    Arecage / Rural


If you are unable to determine the lot size from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
lotsize_predictions = {}

for house_id in range(1,21):

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_1.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_1.jpg': {e}")
        continue

    
    # Initialize list if needed
    if house_id not in lotsize_predictions:
        lotsize_predictions[house_id] = []

    lotsize_predictions[house_id].append(prediction)

    print(house_id, prediction)
    
print("\n\n\n")
print(lotsize_predictions)

In [ ]:
# This cell cleans up the values in the 
# house_predictions dictionary created in the above cell.
cleaned = {}

for key, value in lotsize_predictions.items():
    cleaned[key] = value.pop().strip('"')
  
lotsize_predictions = cleaned

print(lotsize_predictions)

# NOTE Below cell can be removed since now I have a working claude model to query from.

In [ ]:
# Note each entry will have a key to value of form:
# key: house_id_image_id, ie) 1_1 

#lotsize_predictions = {
    "1_1": "Medium",
    "2_1": "Medium",
    "3_1": "Small",
    "4_1": "Medium",
    "5_1": "Medium",
    "6_1": "Medium",
    "7_1": "Medium",
    "8_1": "Small",
    "9_1": "Smal",
    "10_1": "Small",
    "11_1": "Small",
    "12_1": "Medium",
    "13_1": "Small",
    "14_1": "Small",
    "15_1": "Large",
    "16_1": "Small",
    "17_1": "Medium",
    "18_1": "Medium",
    "19_1": "Small",
    "20_1": "Medium"
}



#house_predictions = {
    "1_1": "Modern",
    "2_1": "Modern",
    "3_1": "Modern Farmhouse",
    "4_1": "Two-story Traditional",
    "5_1": "Modern",
    "6_1": "Two-story Traditional",
    "7_1": "Two-story Traditional",
    "8_1": "Two-story Traditional",
    "9_1": "Modern",
    "10_1": "Two-story Traditional",
    "11_1": "Modern",
    "12_1": "Two-story Traditional",
    "13_1": "Bungalow",
    "14_1": "Modern",
    "15_1": "Two-story Traditional",
    "16_1": "Bungalow",
    "17_1": "Two-story Traditional",
    "18_1": "Two-story Traditional",
    "19_1": "Two-story Traditional",
    "20_1": "Two-story Traditional"
}


#luxury_predictions = {
    # House 1
    "1_1": "High End",
    "1_2": "High End",
    "1_3": "High End",
    "1_4": "High End",
    "1_5": "High End",
    "1_6": "Mid",
    "1_7": "Mid",

    # House 2
    "2_1": "High End",
    "2_2": "High End",
    "2_3": "High End",
    "2_4": "High End",
    "2_5": "Mid",
    "2_6": "Mid",

    # House 3
    "3_1": "Mid",
    "3_2": "Mid",
    "3_3": "Mid",
    "3_4": "Mid",
    "3_5": "Basic",

    # House 4
    "4_1": "Mid",     # Exterior
    "4_2": "Mid",     # Kitchen
    "4_3": "Mid",     # Living Room
    "4_4": "Basic",   # Dining Area
    "4_5": "Mid",     # Bedroom
    "4_6": "Mid",     # Bathroom
    "4_7": "Basic",    # Backyard

    # House 5 
    "5_1": "Mid",       # Exterior: modern suburban, nice but not ultra-premium finishes
    "5_2": "High End",  # Kitchen: high-end appliances, large island, custom cabinetry, wine fridge
    "5_3": "High End",   # Dining/living: upscale furniture, open layout, premium finishes
    "5_4": "High End",
    "5_5": "High End",
    "5_6": "Mid",
    "5_7": "High End", 

    # House 6: 
    "6_1": "Mid",
    "6_2": "High End",
    "6_3": "Mid",
    "6_4": "Mid",
    "6_5": "Basic",
    "6_6": "Basic",
    "6_7": "Mid", 

    # House 7
    "7_1": "Mid",
    "7_2": "Mid",
    "7_3": "Basic",
    "7_4": "Basic",
    "7_5": "Basic",
    "7_6": "Basic",
    "7_7": "Mid",

    # House 8
    "8_1": "Mid",
    "8_2": "Mid",
    "8_3": "Mid",
    "8_4": "Mid",
    "8_5": "Mid",
    "8_6": "Mid",

    # House 9
    "9_1": "High End",
    "9_2": "High End",
    "9_3": "High End",
    "9_4": "High End",
    "9_5": "High End",
    "9_6": "High End",
    "9_7": "High End", 

    # House 10:
    "10_1": "High End",
    "10_2": "High End",
    "10_3": "High End",
    "10_4": "High End",
    "10_5": "High End",
    "10_6": "High End",
    "10_7": "High End",

    # House 11
    "11_1": "Mid",
    "11_2": "Mid",
    "11_3": "Mid",
    "11_4": "Mid",
    "11_5": "Mid",
    "11_6": "Mid",

    # House 12
    "12_1": "Mid",
    "12_2": "Mid",
    "12_3": "Mid",
    "12_4": "Mid",
    "12_5": "Mid",
    "12_6": "Mid",
    "12_7": "Mid",

    # House 13
    "13_1": "Mid",
    "13_2": "High End",
    "13_3": "Mid",
    "13_4": "Mid",
    "13_5": "Mid",
    "13_6": "Mid",
    "13_7": "Mid",

    # House 14
    "14_1": "Mid",
    "14_2": "Mid",
    "14_3": "Mid",
    "14_4": "Mid",
    "14_5": "Mid",
    "14_6": "Mid",
    "14_7": "Mid", 

    # House 15
    "15_1": "Mid",
    "15_2": "Mid",
    "15_3": "Mid",
    "15_4": "Mid",
    "15_5": "Mid",
    "15_6": "Mid",
    "15_7": "Mid",

    # House 16
    "16_1": "Mid",
    "16_2": "High End",
    "16_3": "High End",
    "16_4": "High End",
    "16_5": "High End",
    "16_6": "High End",
    "16_7": "Mid", 

    # House 17
    "17_1": "Mid",
    "17_2": "Mid",
    "17_3": "Mid",
    "17_4": "Mid",
    "17_5": "Mid",
    "17_6": "Mid",
    
    # House 18
    "18_1": "Mid",
    "18_2": "Mid",
    "18_3": "Mid",
    "18_4": "Mid",
    "18_5": "Mid",
    "18_6": "Mid",
    "18_7": "Mid",

    # House 19
    "19_1": "Mid",
    "19_2": "Mid",
    "19_3": "Mid",
    "19_4": "Mid",
    "19_5": "Mid",
    "19_6": "Mid",
    "19_7": "Mid",
    
    # House 20
    "20_1": "High End",
    "20_2": "High End",
    "20_3": "High End",
    "20_4": "High End",
    "20_5": "High End"
}

# Section 2: Data Analysis

Here I perform summary statitics, and data visualization to identify patterns anomalies, and relationships between variables.

This is done to give me an understanding of the datas' key characteristics and to prepare it for more complex tasks.


In [ ]:
# This cell will find the most common result for each house in luxury classifications.
# It will give each house a single classification.

luxury_predictions_final = {}

for house_num in range(1, 21):
    house_id = str(house_num)

    # Collect all labels for this house
    labels = []
    for key, value in luxury_predictions.items():
        # key format: "H_I"
        parts = str(key).split("_")
        if len(parts) != 2:
            continue

        if parts[0] == house_id:
            # Handle both string and list values safely
            if isinstance(value, list):
                for v in value:
                    labels.append(str(v))
            else:
                labels.append(str(value))

    # Skip houses with no images
    if not labels:
        continue

    # Count occurrences manually
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    # Find the most common label
    most_common_label = None
    highest_count = -1
    for label, count in counts.items():
        if count > highest_count:
            highest_count = count
            most_common_label = label

    luxury_predictions_final[house_id] = most_common_label

print(luxury_predictions_final)

In [ ]:
# This cell cleans the values for the luxury_predictions_final dict:
luxury_predictions = {}

for key, value in luxury_predictions_final.items():
    if isinstance(value, str):
        luxury_predictions[int(key)] = value.strip('"')
    else:
        luxury_predictions[int(key)] = value

print(luxury_predictions)

Below creates new combined sql table containing all prior stored data and inferences made for each house.

In [ ]:
/*
This table will be used to hold the inferences made for type, luxury and lot size for each house.
*/
CREATE OR REPLACE TEMP TABLE house_inferences (
    house_id INT,
    house_type STRING,
    luxury STRING,
    lot_size STRING 
);


In [ ]:
# I now need to add entries to the above temp table. 

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

for house_id in house_predictions.keys():

    house_type = house_predictions[house_id]
    luxury = luxury_predictions[house_id]
    lot_size = lotsize_predictions[house_id]

    # Clean values (remove extra quotes)
    house_type = str(house_type).strip('"').strip("'")
    luxury = str(luxury).strip('"').strip("'")
    lot_size = str(lot_size).strip('"').strip("'")

    sesh.sql(f"""
        INSERT INTO house_inferences (house_id, house_type, luxury, lot_size)
        VALUES ({house_id}, '{house_type}', '{luxury}', '{lot_size}')
    """).collect()


In [ ]:
SELECT * 
FROM house_inferences;

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()



df_data = sesh.table("house_data")
df_inf = sesh.table("house_inferences")

# Remove image_path and deduplicate house_data
df_data_unique = df_data.select(
    "house_id",
    "address",
    "price",
    "bedrooms",
    "bathrooms",
    "sqft",
    "url_link"
).distinct()

# Join on house_id
df_joined = df_data_unique.join(
    df_inf,
    df_data_unique["house_id"] == df_inf["house_id"]
)

# Select final columns (1 row per house)
df_final = df_joined.select(
    df_data_unique["house_id"],
    df_data_unique["address"],
    df_data_unique["price"],
    df_data_unique["bedrooms"],
    df_data_unique["bathrooms"],
    df_data_unique["sqft"],
    df_data_unique["url_link"],
    df_inf["house_type"],
    df_inf["luxury"],
    df_inf["lot_size"]
)

# Save to final table
sesh.sql("DROP TABLE IF EXISTS all_house_data").collect()
df_final.write.save_as_table("all_house_data", mode="append")

In [ ]:
SELECT * 
FROM all_house_data;

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

house_data_df = sesh.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.ALL_HOUSE_DATA").to_pandas()
house_data_df

In [ ]:
# Below is used to check the types 
# used in the data frame.
house_data_df.dtypes

In [ ]:
# EXPLORATORY DATA ANALYSIS
# Here I perform summary statitics, and 
# data visualization to identify patterns
# anomalies, and relationships between variables. 
# This is done to give me an understanding of 
# the datas' key characteristics and to prepare 
# it for more complex tasks.

# Step 1: Feature Distribution:

# Step 1: Feature Distribution

# TODO move below cell to further down in project where numeric_df is actually used. 

# NOTE: Ask for each cell below if there are any changeed that need to be made for a price predictoin project:

In [ ]:
# Method 1: A Script Based Approach.
import streamlit 
import altair
import pandas as pd

# Below displays a large header at the top of the page. 
streamlit.header("Feature Distributions")

features_df = house_data_df[["PRICE", "BEDROOMS", "BATHROOMS", "SQFT", "LUXURY", "HOUSE_TYPE", "LOT_SIZE"]]

# Below creates a form named distribution_form
# Forms group widgets together so they only trigger when the user presses a submit button.
with streamlit.form("distribution_form"):
    # Below creates a downdown menu, that contains the 
    # column lable sof the dataframe as options.
    feature = streamlit.selectbox("Select a Feature", features_df.columns)
    
    # Below adds a submit button inside of the form.
    submit_dist = streamlit.form_submit_button("Submit", type = "primary")

# Below chekcs if the form was submitted.
if submit_dist:

    # Below checks if the selected column is numeric. 
    if pd.api.types.is_numeric_dtype(features_df[feature]):
        # If a numeric column was selected, create a histogram
        chart = altair.Chart(features_df).mark_bar().encode(
            # Below creates a chart using the whole DataFrame
            altair.X(f"{feature}:Q", bin = True),
            # tells Altair to draw bars
            y = 'count()',
            # Below colors bars based on the feature value
            # which is converted to nominal (ie. N)
            # scale controls how values map to colors, and below 
            # we use Altair's browns color palette.
            color =  altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme = "blues"),
                    legend = None
                )
        )
        
    else:
        # If a non-numeric column was selected, create a categorical bar chart
        chart = altair.Chart(features_df).mark_bar().encode(
            x =  altair.X(f"{feature}:N"),
            y = 'count()',
            color = altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme = "blues"),
                    legend = None
                )
        )
    # Below displays the Altair chart in Streamlit. 
    # Properties sets some styling properties for the chart. 
    # use_Container_width=True makes the chart responsive.
    streamlit.altair_chart(chart.properties(height = 380, title = f"Distribution of {feature}"), use_container_width = True)
     

# STEP 2: FEATURE CORRELATIONS

In [ ]:
# Below displays a title.
streamlit.title("🏠 House Features Explorer")

# Load your DataFrame
# house_data_df = session.table("SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_DATA_DF").to_pandas()

# For now assume df_raw already exists
streamlit.subheader("Dataset Preview") # shows a subheader
streamlit.dataframe(house_data_df.head()) # Creates a datafreame in streamlit

# Correct column type detection
# Below selects the numeric columns from the data frame.
numeric_cols = house_data_df[["PRICE", "BEDROOMS", "BATHROOMS", "SQFT"]].columns.tolist()

# Below selects the non-numeric columns from the data frame.
categorical_cols = house_data_df[["LUXURY", "HOUSE_TYPE", "LOT_SIZE"]].columns.tolist()


streamlit.markdown("### Column Types") # Displays a markdown header
streamlit.write("**Numeric columns:**", numeric_cols) # Writes the numeric columns headings to outout.
streamlit.write("**Categorical columns:**", categorical_cols) # Writes the non-numeric columns headings to output.
 
# -----------------------------
# FEATURE DISTRIBUTION SECTION
# -----------------------------
streamlit.header("📈 Feature Distributions")


# Below creates a form so that the user must click a button to trigger the chart.  
with streamlit.form("feature_form"):
    # Below creates a a select box that contains the data frames column titles. 
    feature = streamlit.selectbox("Select a Feature", features_df.columns)
    #Below creates a submit button.
    submitted = streamlit.form_submit_button("Show Distribution", type = "primary")

# Check if the submit button was pressed.
if submitted:
    # Display a sub header, with the selected feature displayed.
    streamlit.subheader(f"Distribution of `{feature}`")

    # If a numeric Feature was selected → An Altair histogram is displayed
    if feature in numeric_cols:
        chart = (
            altair.Chart(house_data_df)
            .mark_bar()
            .encode(
                altair.X(f"{feature}:Q", bin=altair.Bin(maxbins = 30)),
                y = "count()"
            )
            .properties(height = 350)
        )
        streamlit.altair_chart(chart, use_container_width = True)

    # If a categorical feature was selected → An Altair count plot is displayed.
    elif feature in categorical_cols:
        chart = (
            altair.Chart(house_data_df)
            .mark_bar()

            
            .encode(
                x = altair.X(f"{feature}:N", sort='-y'),
                y = "count()",
                
                color = altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme="blues"),
                    legend = None
                )

            )
            .properties(height = 350)
        )
        streamlit.altair_chart(chart, use_container_width = True)

    # If neither a numeric or non-numeric feature is selected, a warning message is displayed. 
    else:
        streamlit.warning("This feature type is not supported yet.")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
streamlit.header("📊 Summary Statistics")

col1, col2 = streamlit.columns(2) # Creates 2 column containers 

# In column container 1 display the numeric data summary  
with col1:
    streamlit.markdown("### Numeric Summary")
    streamlit.write(house_data_df[numeric_cols].describe())


# In column container 2 display the non-numeric data summary
with col2:
    streamlit.markdown("### Categorical Summary")
    streamlit.write(house_data_df[categorical_cols].describe())

In [ ]:
import numpy  as np
import pandas as pd

def correlation_ratio(categories, values):
    """
    correlation_ratio:
        This function takes in a categorical feature and a a numeric feature and
        it returns a measure of how much of the variance in values is explained 
        by categories.
    """

    # Converts categories into pandas categorical object. 
    categories = pd.Categorical(categories)
    # Extracts the unique category levels. 
    cat_levels = categories.categories
    # Compuates the overall mean of the given numeric category saved in the variable values.
    y_mean = values.mean()

    # Below builds the between-group sum of squares:
    # For each category cat in cat_levels: 
    # values[categories==cat] shows all numeric values belonging in that category. 
    numerator = sum([
        # len(... is the number of observations in that category.
        len(values[categories == cat]) *
        # Below si the squared difference between that category's mean and the overall mean. 
        (values[categories == cat].mean() - y_mean) ** 2
        for cat in cat_levels
    ])
    # Sum above adds the quantity calculated within over all categories.

    # Below is total sum of squares / AKA the total variance in the numeric variable:
    denominator = sum((values - y_mean) ** 2)
    
    # below cpmputes and returns the correlation ratio.
    # If denominator is not equal to 0, compute the proportion of variance explained by categories.
    # If denominator is equal to zero, all values are identical (no variance) and thus we return 0.
    return np.sqrt(numerator / denominator) if denominator != 0 else 0


def mixed_correlation(df):
    """
    mixed_correlation:
        Takes in a pandas dataframe and returns a correlation matrix that 
        works for numeric, categorial, and mixed pairs.
    """
    
    cols = df.columns # Stores columns in the dataframe.
    # Creates an empty square dataframe to store the results. 
    # rows/index as well as the columns of this new dataframe 'corr' are set 
    # to the columns of the 'cols' dataframe. All values are also set to floats.
    corr = pd.DataFrame(index = cols, columns = cols, dtype = float) 

    # Below loops over every pair of columns. 
    # This ensures a full matrix.
    for col1 in cols:
        for col2 in cols:
            # If the 2 columns are the same.
            if col1 == col2:
                # Set the correlation to 1, and skip the rest of the loop.
                corr.loc[col1, col2] = 1.0
                continue

            # Below extracts the 2 columns being compared.
            x = df[col1]
            y = df[col2]

            # Numeric ↔ Numeric
            # If both columns are numeric, we use a Pearson correlation, and store the results in the matrix.
            if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
                corr.loc[col1, col2] = x.corr(y)

            # Categorical ↔ Categorical
            # If both columns are non-numeric, we use a Cramer's V correlation, and store the results in the matrix.
            elif x.dtype == "object" and y.dtype == "object":
                corr.loc[col1, col2] = cramers_v(x, y)

            # Numeric ↔ Categorical
            # If one category is numeric and the other is non-numeric, this checks which is 
            # numeric and orders them correctly for the n_r
            else:
                if pd.api.types.is_numeric_dtype(x):
                    corr.loc[col1, col2] = correlation_ratio(y, x)
                else:
                    corr.loc[col1, col2] = correlation_ratio(x, y)

    return corr # Return the correlation matrix.


In [ ]:
"""
Cramér’s V is a correlation measure for 
categorical variables. 

This measure us used to describe the relationship
between two non-numeric features 
   
Possible return values from a cramers analysis:
    0.0 -- No association at all. 
    0.1-0.3 -- Weak association.
    0.3-0.6 -- Moderate association. 
    0.6-1.0 -- Strong association.

How does it work?
    1) Build a contingency table: 
        Which is like a pivot table 
        counting combinations of categories 

    2) Run a chi-square test on that table 
        Measures how different the observed counts
        are from random chance. 

    3) Normalize the chi-square statistic 
        This allows the result to always be beween 
        0 and 1.

This us a useful meausre since categorical features 
don't have numeric distance, thus you can't compute 
a Pearson correlation. 

"""

from scipy.stats import chi2_contingency


def cramers_v(x, y):
    """
    Calculates cramer's V given the 2 input variables, x and y. This is a 
    measure of how closely related the 2 variables are. 

    x and y are two different columns from the dataframe being processed 
    in the function categorical_correlation. 
    """
    
    confusion_matrix = pd.crosstab(x, y) # Create contingency table.

    # If either variable has only one category → no association
    if confusion_matrix.shape[0] == 1 or confusion_matrix.shape[1] == 1:
        return 0.0

    chi2 = chi2_contingency(confusion_matrix)[0] # runs chi square test and get chi square value.
    n = confusion_matrix.sum().sum() # holds number of observations.
    phi2 = chi2 / n # calculates chi-square value.
    r, k = confusion_matrix.shape # r is number of categroeis in x, and k is number in y.

    # Bias correction
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1)) # correct chi-square for small sample size 
    rcorr = r - ((r - 1) ** 2) / (n - 1) # Corrects effective number of categories
    kcorr = k - ((k - 1) ** 2) / (n - 1) # Corrects effective number of categories

    # Below calculates the denominatorand then checks if it is invlaid. 
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return 0.0 # If denominator is invalid, return 0.

    return np.sqrt(phi2corr / denom) # Retuens the cramers V value between 0 and 1. 

# Below creates a full correlation matrix for categorical features only.
def categorical_correlation(df):
    """
    Takes a dataframe as input and computes Cramer's V for every pair of columns.
    """
    
    cols = df.columns # Holds a list of the dataframes column names.
    corr = pd.DataFrame(index = cols, columns = cols, dtype = float) # Creates an empty square matrix to fill with Cramers V values.

    # Below loops over every pair of values between column 1 and column 2.
    for col1 in cols:
        for col2 in cols:
            # If column 1 and column 2 are equal (In the diagonal), the feature is said to be perfectly correlated with itself. 
            if col1 == col2:
                corr.loc[col1, col2] = 1.0
            else:
                corr.loc[col1, col2] = cramers_v(df[col1], df[col2])

    return corr # Returns the correlation matrix.


In [ ]:
# Method 1: Script-based approach


corr_matrix = mixed_correlation(features_df) # Gets the correlation matrix of the dataframe.

# 
corr_data = (
    corr_matrix
    .stack() # Converts matrix to series with a multi-index.
    .reset_index(name = 'value') # Turns serries into a dataframe
    # Below renames the dataframes' index coumns 
    .rename(columns = {'level_0': 'feature1', 'level_1': 'feature2'}) 
)


# 
corr_chart = (
    altair.Chart(corr_data) # creates altair chart using the corr_data as the source.

    # Converts each row in corr_data to a rectangle,
    # each of which represent a cell in the heatmap
    .mark_rect() 

    # encode defines how data columns map to visual properties. 
    .encode(
        # Sets feature 1 to the x axis.
        x = altair.X('feature1:N', sort=None),
        # Sets feature 2 to the y axis.
        y = altair.Y('feature2:N', sort=None),
        # sets the color of the correlation value. (uses )
        color = altair.Color('value:Q', scale = altair.Scale(scheme = 'cividis')),
        # Createa a tooltip that displays the naems of both features and its value 
        # when you hover over a cell in the matrix.
        tooltip = [
            alt.Tooltip('feature1:N'),
            alt.Tooltip('feature2:N'),
            alt.Tooltip('value:Q', format = ".3f")
        ]
    )
        
    # Sets the chart width and height and gives it a title..
    .properties(
        width = 500,
        height = 500,
        title = "Mixed-Type Feature Correlation Heatmap"
    )
)

# Displays the Altair chart in the streamlit application.
streamlit.altair_chart(corr_chart, use_container_width = True)



# Results? 
# TODO!!!!!!     

In [ ]:
# Below shows the number of unique data values 
# in each of the columns.
house_data_df.nunique()

# House Type Class Distribution:

This analysis will uncover: The number of unique house types, and the relative positions of each ype in the sample.

In [ ]:
import altair as alt
import streamlit as st

st.header("🏠 House Type Distribution")

st.write("""
Understanding the distribution of house types helps us see:
- Whether the dataset is balanced  
- Which categories dominate  
- How much training data each class contributes  
""")

# Ensure correct column name + clean missing values
df = house_data_df.copy()
df["HOUSE_TYPE"] = df["HOUSE_TYPE"].fillna("Unknown")

# --- BAR CHART ---
bar_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X("HOUSE_TYPE:N", sort='-y'),
        y="count()",
        color=alt.Color("HOUSE_TYPE:N", scale=alt.Scale(scheme="cividis")),
        tooltip=["HOUSE_TYPE:N", alt.Tooltip("count()", title="Count")]
    )
    .properties(
        height=350,
        title="Bar Chart: House Type Distribution"
    )
)

# --- PIE CHART ---
pie_chart = (
    alt.Chart(df)
    .mark_arc()
    .encode(
        theta="count()",
        color=alt.Color("HOUSE_TYPE:N", scale=alt.Scale(scheme="cividis")),
        tooltip=["HOUSE_TYPE:N", alt.Tooltip("count()", title="Count")]
    )
    .properties(
        height=350,
        title="Pie Chart: House Type Proportions"
    )
)

col1, col2 = st.columns(2)
col1.altair_chart(bar_chart, use_container_width=True)
col2.altair_chart(pie_chart, use_container_width=True)

# Luxury Level Distribution:
This analysis will uncover: The number of unique luxury levels assigned, and the relative positions of each luxury levels assigned in the sample.

In [ ]:
import altair as alt
import streamlit as st

st.header("House Lot Size Distribution")

st.write("""
Let's examine the distribution of house lot size categories in our dataset. This helps us understand:
- How many samples exist for each lot size category  
- Whether the dataset is balanced or imbalanced  
- The relative proportions of each lot size category in the overall sample  
""")


actual_col = None
for name in house_data_df.columns:
    cleaned = name.strip().upper().replace(" ", "_")
    if cleaned == "LOT_SIZE":
        actual_col = name
        break

if actual_col is None:
    st.error("No lot size column found in the dataset.")
else:
    df = house_data_df.copy()

    # Clean missing values
    df[actual_col] = df[actual_col].fillna("Unknown")

    # --- BAR CHART ---
    bar_chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=f'{actual_col}:N',
            y='count()',
            color=alt.Color(f'{actual_col}:N', scale=alt.Scale(scheme='cividis')),
            tooltip=[f'{actual_col}:N', alt.Tooltip('count()', title='Count')]
        )
        .properties(
            height=350,
            title="Bar Chart: Lot Size Distribution"
        )
    )

    # --- PIE CHART ---
    pie_chart = (
        alt.Chart(df)
        .mark_arc()
        .encode(
            theta='count()',
            color=alt.Color(f'{actual_col}:N', scale=alt.Scale(scheme='cividis')),
            tooltip=[f'{actual_col}:N', alt.Tooltip('count()', title='Count')]
        )
        .properties(
            height=350,
            title="Pie Chart: Lot Size Proportions"
        )
    )

    col1, col2 = st.columns(2)
    col1.altair_chart(bar_chart, use_container_width=True)
    col2.altair_chart(pie_chart, use_container_width=True)

# Lot Size Distribution:

In [ ]:
import altair as alt
import streamlit as st

st.header("House Luxury Level Distribution")

st.write("""
Let's examine the distribution of house luxury levels assigned in our dataset. This helps us understand:
- How many samples exist for each possible level  
- Whether the dataset is balanced or imbalanced  
- The relative proportions of each level in the overall sample  
""")

# Ensure correct column name + clean missing values
df = house_data_df.copy()
df["LUXURY"] = df["LUXURY"].fillna("Unknown")

# --- BAR CHART ---
bar_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x='LUXURY:N',
        y='count()',
        color=alt.Color('LUXURY:N', scale=alt.Scale(scheme='cividis')),
        tooltip=['LUXURY:N', alt.Tooltip('count()', title='Count')]
    )
    .properties(
        height=350,
        title="Bar Chart: Luxury Distribution"
    )
)

# --- PIE CHART ---
pie_chart = (
    alt.Chart(df)
    .mark_arc()
    .encode(
        theta='count()',
        color=alt.Color('LUXURY:N', scale=alt.Scale(scheme='cividis')),
        tooltip=['LUXURY:N', alt.Tooltip('count()', title='Count')]
    )
    .properties(
        height=350,
        title="Pie Chart: Luxury Level Proportions"
    )
)

# Display side-by-side
col1, col2 = st.columns(2)
col1.altair_chart(bar_chart, use_container_width=True)
col2.altair_chart(pie_chart, use_container_width=True)

In [ ]:
import streamlit as st
import altair as alt
import pandas as pd

st.header("📊 Dataset Exploratory Data Analysis")

# -----------------------------------------
# 1. BASIC OVERVIEW
# -----------------------------------------
st.subheader("Dataset Overview")

df = house_data_df.copy()

# Fix column names (Snowflake → uppercase)
df.columns = [c.upper() for c in df.columns]

# Clean numeric columns
numeric_cols = ["PRICE", "SQFT", "BEDROOMS", "BATHROOMS"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Clean categorical columns
cat_cols = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

st.write(f"Total Samples: {len(df)}")
st.write(f"Numeric Features: {numeric_cols}")
st.write(f"Categorical Features: {cat_cols}")

st.write(f"Number of House Types: {df['HOUSE_TYPE'].nunique()}")
st.write(f"Number of Luxury Levels: {df['LUXURY'].nunique()}")
st.write(f"Number of Lot Size Categories: {df['LOT_SIZE'].nunique()}")

# -----------------------------------------
# 2. SUMMARY STATISTICS
# -----------------------------------------
st.subheader("Summary Statistics")

st.write(f"Average House Price: ${df['PRICE'].mean():,.0f}")
st.write(f"Average Bedrooms: {df['BEDROOMS'].mean():.2f}")
st.write(f"Average Bathrooms: {df['BATHROOMS'].mean():.2f}")
st.write(f"Average Square Footage: {df['SQFT'].mean():.2f}")

st.write(f"Most Common Lot Size: {df['LOT_SIZE'].mode()[0]}")
st.write(f"Most Common Luxury Level: {df['LUXURY'].mode()[0]}")
st.write(f"Most Common House Type: {df['HOUSE_TYPE'].mode()[0]}")

# -----------------------------------------
# 3. PRICE BINNING
# -----------------------------------------
st.subheader("Price Bins")

df["PRICE_BIN"] = pd.cut(
    df["PRICE"],
    bins=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

price_bin_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x="PRICE_BIN:N",
        y="count()",
        color="PRICE_BIN:N"
    )
    .properties(height=300, width=400, title="Distribution of Price Bins")
)

st.altair_chart(price_bin_chart, use_container_width=True)

# -----------------------------------------
# 4. FEATURE DISTRIBUTIONS BY PRICE BIN
# -----------------------------------------
st.subheader("Feature Distributions by Price Range")

for feature in numeric_cols:
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(f"{feature}:Q", bin=True),
            y="count()",
            color=alt.Color("PRICE_BIN:N", scale=alt.Scale(scheme="viridis")),
            tooltip=[feature, "PRICE_BIN"]
        )
        .properties(height=300, width=400, title=f"{feature} Distribution by Price Range")
    )
    st.altair_chart(chart, use_container_width=True)

# -----------------------------------------
# 5. CORRELATION HEATMAP
# -----------------------------------------
st.subheader("Correlation Heatmap")

corr = df[numeric_cols].corr()

corr_data = (
    corr.stack()
    .reset_index(name="value")
    .rename(columns={"level_0": "Feature1", "level_1": "Feature2"})
)

corr_chart = (
    alt.Chart(corr_data)
    .mark_rect()
    .encode(
        x="Feature1:N",
        y="Feature2:N",
        color=alt.Color("value:Q", scale=alt.Scale(scheme="cividis")),
        tooltip=["Feature1", "Feature2", "value"]
    )
    .properties(width=400, height=400, title="Correlation Heatmap")
)

st.altair_chart(corr_chart, use_container_width=True)

# -----------------------------------------
# 6. SCATTERPLOTS (PRICE vs each numeric feature)
# -----------------------------------------
st.subheader("Feature Relationships (Scatterplots)")

for feature in numeric_cols:
    if feature != "PRICE":
        scatter = (
            alt.Chart(df)
            .mark_circle(size=60, opacity=0.5)
            .encode(
                x="PRICE:Q",
                y=f"{feature}:Q",
                color=alt.Color("PRICE:Q", scale=alt.Scale(scheme="viridis")),
                tooltip=["PRICE", feature]
            )
            .properties(width=380, height=380, title=f"PRICE vs {feature}")
        )
        st.altair_chart(scatter, use_container_width=True)

# -----------------------------------------
# 7. CATEGORICAL DISTRIBUTIONS
# -----------------------------------------
st.subheader("Categorical Feature Distributions")

for col in cat_cols:
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=f"{col}:N",
            y="count()",
            color=f"{col}:N"
        )
        .properties(width=380, height=380, title=f"Distribution of {col}")
    )
    st.altair_chart(chart, use_container_width=True)

# Section 3 - Part 1: Data Operations

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

house_data_df2 = sesh.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.ALL_HOUSE_DATA").to_pandas()
house_data_df2

Since predicting house price is a regression problem (not a classification problem, I can't use the price bins created during section 2)

In [ ]:
# Below prepares the features and price:
# Dataframe is separated into 2. Features are assigned to the x variable 
# whlie the price is assigned to y.
# Prepare features and target

numeric_cols = ["SQFT", "BEDROOMS", "BATHROOMS"]
categorical_cols = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]

X = house_data_df2[numeric_cols + categorical_cols]
y = house_data_df2["PRICE"]


# Above sets us up for a regression model.

In [ ]:
# Check for missing data: 

# Data quality checks
# X is the feature matrix, AKA all the columns used to train a model.
# .isnull() creates a dataframe of True/Flase values where True used 
# when value is missing, and false used when value is present.
# Use .sum().sum() as this sums columns wise, giving num missing vals per collumn,
# y is the target variable, AKA the thing we are trying to predict. 
# isnull().sum() counts how many missing values exist in the target column.
missing_features = X.isnull().sum().sum() 
missing_target = y.isnull().sum()

print(f"\n🔍 Data Quality:")
print(f"   Missing feature values: {missing_features}")
print(f"   Missing target values: {missing_target}")

In [ ]:
# Data Splitting:
# Data is separated into training-testing sets using 80/20 ratio in scikit-learn
# Preserves class balance usig stratification. 
# Ensures reproducibility with a fixed random seed
# Prints a clean summary of: number of samples in each split, number of features, and the class distribution.

# Import scikit-learn modules at first use
from sklearn.model_selection import train_test_split

# Below splits data using scikit-learn (recommended by Snowflake for ML operations)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, # X is the feature matrix. y is the target labels.
    test_size = 0.2, # 20% of the data is the test set, thus 80% is the training set.
    random_state = 42, # This ensures the split is reproducible.
    # No stratify for regression
)

print("✅ Data splitting completed!")
print('-' * 35) 

print("📊 Data Split Summary:")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Number of features: {X_train.shape[1]}")
print('-' * 35)

print("\n🎯 Target Summary:")
print(f"Training target stats: min={y_train.min():.2f}, max={y_train.max():.2f}, mean={y_train.mean():.2f}")
print(f"Testing target stats: min={y_test.min():.2f}, max={y_test.max():.2f}, mean={y_test.mean():.2f}")
print('-' * 35) 

In [ ]:
# DATA PREPARATION:

# Select only numeric columns from the DataFrame
numeric_df = house_data_df[["PRICE", "BEDROOMS", "BATHROOMS", "SQFT"]]

numeric_df

"""
Other possible data types to select: 
object, 
category, 
datetime,
number.
"""
     

In [ ]:
# Feature Scaling: 
# This is a data preprocessing technique used to standardize the range of 
# indenpendent variables (AKA features) of the data sample. This helps to ensure 
# that features with larger value ranges do not disporportionately affect the model's 
# learning process.

# Process of this cell:
# ID numerical vs categorical features
# Scale numerical features
# Encode categorical features 
# Recombine everything into a clean, final dataframe.


# We use scikit-learn to do this with 'mean centering' (mean = 0) unit variance (SD=1) 
# This sets all variables' means and SD to 0 and 1, respectively. 

# ---------------------------------------------------------
# FEATURE SCALING + ENCODING (FULLY FIXED VERSION)
# ---------------------------------------------------------

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("============================================")
print("Feature Scaling & Encoding")
print("============================================")

# Ensure X_train and X_test exist and are not empty
if X_train is None or X_train.empty:
    raise ValueError("❌ X_train is empty or not defined")

if X_test is None or X_test.empty:
    raise ValueError("❌ X_test is empty or not defined")

# Identify numeric + categorical columns
numerical_features = X_train.select_dtypes(include=['int8', 'int16', 'int32']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Safety check: avoid empty arrays
if len(numerical_features) == 0 and len(categorical_features) == 0:
    raise ValueError("❌ No numeric or categorical features found in X_train")

# ---------------------------------------------------------
# SCALE NUMERIC FEATURES
# ---------------------------------------------------------
if len(numerical_features) > 0:
    scaler = StandardScaler()
    X_train_scaled_num = scaler.fit_transform(X_train[numerical_features])
    X_test_scaled_num = scaler.transform(X_test[numerical_features])
else:
    X_train_scaled_num = np.empty((len(X_train), 0))
    X_test_scaled_num = np.empty((len(X_test), 0))

# ---------------------------------------------------------
# ENCODE CATEGORICAL FEATURES
# ---------------------------------------------------------
if len(categorical_features) > 0:
    onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_train_scaled_cat = onehot.fit_transform(X_train[categorical_features])
    X_test_scaled_cat = onehot.transform(X_test[categorical_features])
    cat_feature_names = onehot.get_feature_names_out(categorical_features)
else:
    X_train_scaled_cat = np.empty((len(X_train), 0))
    X_test_scaled_cat = np.empty((len(X_test), 0))
    cat_feature_names = []

# ---------------------------------------------------------
# COMBINE NUMERIC + CATEGORICAL
# ---------------------------------------------------------
X_train_scaled = np.hstack([X_train_scaled_num, X_train_scaled_cat])
X_test_scaled = np.hstack([X_test_scaled_num, X_test_scaled_cat])

# ---------------------------------------------------------
# CONVERT BACK TO DATAFRAME
# ---------------------------------------------------------
all_feature_names = numerical_features + list(cat_feature_names)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=all_feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=all_feature_names, index=X_test.index)

print("\n✅ Feature scaling completed!")
print("-----------------------------------")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)
print("-----------------------------------")

# ---------------------------------------------------------
# SHOW SCALING EFFECT
# ---------------------------------------------------------
if len(numerical_features) > 0:
    first_num = numerical_features[0]
    print("\n📊 Scaling Effect (first numeric feature):")
    print(f"Original {first_num}: mean={X_train[first_num].mean():.3f}, std={X_train[first_num].std():.3f}")
    print(f"Scaled   {first_num}: mean={X_train_scaled[first_num].mean():.3f}, std={X_train_scaled[first_num].std():.3f}")

print("-----------------------------------")

 # Section 3 - Part 2: Machine Learning Model Training

# Some modelees that would be good for cointuous price predictoin: 
- RandomForestRegressor
- Linear Regression 
- XGBoost Regressor 

In [ ]:
# Regression Model Training and Evaluation of the Random Forest Regressor Model

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🔧 Training Random Forest Regressor...")

# Create and train the model
reg_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
reg_model.fit(X_train_scaled, y_train)

# Make predictions
rf_train_pred = reg_model.predict(X_train_scaled)
rf_test_pred = reg_model.predict(X_test_scaled)

# Compute regression metrics
rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)


print("✅ Regression model trained!")
print('-' * 35)

print("📊 Regression Model Results:")
print(f"   Training MAE:  {rf_train_mae:.2f}")
print(f"   Testing MAE:   {rf_test_mae:.2f}")
print(f"   Training RMSE: {rf_train_rmse:.2f}")
print(f"   Testing RMSE:  {rf_test_rmse:.2f}")
print(f"   Training R²:   {rf_train_r2:.4f}")
print(f"   Testing R²:    {rf_test_r2:.4f}")
print('-' * 35)

We see that the above random forest model is performing very accurately (At 99.99%) and this expected considering the sample we have used.
	- Random forest are comprised of decision trees, and each tree is trained on a random subset of rows and features. This allows the model to capture nonlinear relationships, as well as interactions between categorial features. 
	- But since the sample size is so small, the model is able to near perfectly memorize the combination of features that each house has. This allows the model to predict the homes price, based on that combination.	

	- I can try to alter the hyperparameters to get more realistic results, and I do so in the below cell. 
		I decreased the number of n_estimators (number of trees) to 150, 
		Limited the depth of each tree to 4 nodes, 
		Set the minimum number of samples needed to split a node to 10 (prevents trees from splitting on tiny groups of homes), 
		Set the minimum samples in a leaf to 4 (larger leaves means less memorization), 
		Set the number of features each tree can use at each split to sqrt(number of features) and then it chooses the best split among them (increases randomness and reduces overfitting).


And we can see that the prediction values are made to be a bit more realistic (and less accurate)

In [ ]:
# Regression Model Training and Evaluation of the Random Forest Regressor Model

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🔧 Training Random Forest Regressor...")

# Create and train the model
reg_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42
)


reg_model.fit(X_train_scaled, y_train)

# Make predictions
train_pred = reg_model.predict(X_train_scaled)
test_pred = reg_model.predict(X_test_scaled)

# Compute regression metrics
rf_train_pred = reg_model.predict(X_train_scaled)
rf_test_pred = reg_model.predict(X_test_scaled)

# Compute regression metrics
rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)

print("✅ Regression model trained!")
print('-' * 35)

print("📊 Regression Model Results:")
print(f"   Training MAE:  {rf_train_mae:.2f}")
print(f"   Testing MAE:   {rf_test_mae:.2f}")
print(f"   Training RMSE: {rf_train_rmse:.2f}")
print(f"   Testing RMSE:  {rf_test_rmse:.2f}")
print(f"   Training R²:   {rf_train_r2:.4f}")
print(f"   Testing R²:    {rf_test_r2:.4f}")
print('-' * 35)

In [ ]:
# This cell trains a K-Nearest Neighbors Regression model 

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🔧 Training K-Nearest Neighbors Regressor...")
print('-' * 35)

# Initialize the model
knn_model = KNeighborsRegressor(
    n_neighbors=5,      # You can tune this later
    weights='distance'  # Distance-weighted predictions often perform better
)

# Train the model
knn_model.fit(X_train_scaled, y_train)

# Predictions
knn_train_pred = knn_model.predict(X_train_scaled)
knn_test_pred = knn_model.predict(X_test_scaled)

# Metrics
knn_train_mae = mean_absolute_error(y_train, knn_train_pred)
knn_test_mae = mean_absolute_error(y_test, knn_test_pred)

knn_train_rmse = np.sqrt(mean_squared_error(y_train, knn_train_pred))
knn_test_rmse = np.sqrt(mean_squared_error(y_test, knn_test_pred))

knn_train_r2 = r2_score(y_train, knn_train_pred)
knn_test_r2 = r2_score(y_test, knn_test_pred)

print("✅ KNN model trained!")
print('-' * 35)

print("📊 KNN Regression Results:")
print(f"   Training MAE:  {knn_train_mae:.2f}")
print(f"   Testing MAE:   {knn_test_mae:.2f}")
print(f"   Training RMSE: {knn_train_rmse:.2f}")
print(f"   Testing RMSE:  {knn_test_rmse:.2f}")
print(f"   Training R²:   {knn_train_r2:.4f}")
print(f"   Testing R²:    {knn_test_r2:.4f}")
print('-' * 35)

# Note: 

The above results show that the KNN nodel accurately predicts the sample house prices with 100% accuracy --> This is impossible!! 

Whats going on? 


Well the given combination of features createa a unique fingerprint for each house, and this allows the model to recognize when a select grouping of features is presetn, and thus can accuratley ID the hosue price.

To allow KNn to work more realistically, I am going to make a number of changes: 

Removal of weights="distance" --> this prevents infinte weighting of indetical rows

Increase K dramatically 
which forces the model to average across multiple neightbors

and reduce categorical granularity
The categories are extremely specific currently (6+ house types, 3 luxury levels, 2 lot sizes) hen these get combined with nuemriical featueess this creates a unique rows! These thus allowthe model to ID the house, and thus the price with 100% accuracy.

, 

In [ ]:
# This cell trains a K-Nearest Neighbors Regression model 

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🔧 Training K-Nearest Neighbors Regressor...")
print('-' * 35)

# Initialize the model
knn_model = KNeighborsRegressor(
    n_neighbors=20      # You can tune this later
)

# Train the model
knn_model.fit(X_train_scaled, y_train)

# Predictions
knn_train_pred = knn_model.predict(X_train_scaled)
knn_test_pred = knn_model.predict(X_test_scaled)

# Metrics
knn_train_mae = mean_absolute_error(y_train, knn_train_pred)
knn_test_mae = mean_absolute_error(y_test, knn_test_pred)

knn_train_rmse = np.sqrt(mean_squared_error(y_train, knn_train_pred))
knn_test_rmse = np.sqrt(mean_squared_error(y_test, knn_test_pred))

knn_train_r2 = r2_score(y_train, knn_train_pred)
knn_test_r2 = r2_score(y_test, knn_test_pred)

print("✅ KNN model trained!")
print('-' * 35)

print("📊 KNN Regression Results:")
print(f"   Training MAE:  {knn_train_mae:.2f}")
print(f"   Testing MAE:   {knn_test_mae:.2f}")
print(f"   Training RMSE: {knn_train_rmse:.2f}")
print(f"   Testing RMSE:  {knn_test_rmse:.2f}")
print(f"   Training R²:   {knn_train_r2:.4f}")
print(f"   Testing R²:    {knn_test_r2:.4f}")
print('-' * 35)

I changed k = 20 (look sat 20 closest homes) and thus the KNN algorthim now predicts the price as being the average price of 20 somehwat similar homes (which just so happens to be the size of the sampe)

this smoothes out our predictis, and reduces the prio rproblem of overfiting. This also icnreases error though, as the model now becomes less precise! 

In [ ]:
# This cell trains a XGBoost Regression model

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🚀 Training XGBoost Regressor...")
print('-' * 35)

# Initialize the model
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

# Train the model
xgb_model.fit(X_train_scaled, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train_scaled)
xgb_test_pred = xgb_model.predict(X_test_scaled)

# Metrics
xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)

xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))

xgb_train_r2 = r2_score(y_train, xgb_train_pred)
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print("✅ XGBoost model trained!")
print('-' * 35)

print("📊 XGBoost Regression Results:")
print(f"   Training MAE:  {xgb_train_mae:.2f}")
print(f"   Testing MAE:   {xgb_test_mae:.2f}")
print(f"   Training RMSE: {xgb_train_rmse:.2f}")
print(f"   Testing RMSE:  {xgb_test_rmse:.2f}")
print(f"   Training R²:   {xgb_train_r2:.4f}")
print(f"   Testing R²:    {xgb_test_r2:.4f}")
print('-' * 35)

# Note: Above cell shows that the XGBoost model can predict the house prices with 100% accuracy (R^2 is 1.0000) 
Why?
This XGBoost model is a very high capacity model (500 trees, depth of 6, subsample of 0.8, colsample of 0.8, nadn learning rate of 0.05) andbeinbg that the dataset has only 105 training rows,  the model can memorize the entire dataset, training set, and generalize it perfecrly y tyo tyhe test set if the test set contains houses tht are similar to the training set. Which it does!

We can tune the hyperparameters as  such, to give more realistic results: 

Reduce tree depth 
Reduce the number of trees, 
Increase Regularization 
Increase sunsmapling, or even add more data (add )


I do the above changes in the below cell: 

In [ ]:
# This cell trains a XGBoost Regression model

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("🚀 Training XGBoost Regressor...")
print('-' * 35)

# Initialize the model
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.6,
    colsample_bytree=0.6,
    random_state=42,
    reg_alpha=1.0,
    reg_lambda=1.0,
    objective='reg:squarederror'
)

# Train the model
xgb_model.fit(X_train_scaled, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train_scaled)
xgb_test_pred = xgb_model.predict(X_test_scaled)

# Metrics
xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)

xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))

xgb_train_r2 = r2_score(y_train, xgb_train_pred)
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print("✅ XGBoost model trained!")
print('-' * 35)

print("📊 XGBoost Regression Results:")
print(f"   Training MAE:  {xgb_train_mae:.2f}")
print(f"   Testing MAE:   {xgb_test_mae:.2f}")
print(f"   Training RMSE: {xgb_train_rmse:.2f}")
print(f"   Testing RMSE:  {xgb_test_rmse:.2f}")
print(f"   Training R²:   {xgb_train_r2:.4f}")
print(f"   Testing R²:    {xgb_test_r2:.4f}")
print('-' * 35)

why changing the hyperparamters makes a differene? 
Decrease max depth makes trees simpler, and thus they cant memoruze exact patterns but will only learn borad patterns.

Why decreasing subsampling to 60% this says each tre can only see 60% of the training rows, and thus the model cant memorize the entire dataset. )

 
But eh predictionsa re still are still very high in the altered cell above though!! This is due to the nature of the algoriothm couple with the size of the dataset. Also the highly structured cateogriclal features create tight cluisters that lead to over over generalization ( I THINK)


# Section 3 - Part 3: Benchmarking of Machine Learning Algorithms

Benchmarking means we are comparing various ML models to see which perform the best and/or are best for the specific use case.

We wish to seelct a model that can generalize well on new, unseen data and one that can provide actionable insights.

We are careful to be aware of the following 2 points:

Model Overfitting: We can evaluate how well the model generalizes on new data by testing the degree the algorithm overfits the data.
Model Interpretability: We ensure our model yields actional insight by understanding the underlying reasons as to 'why' the model made the predictions it does. We can do this by tracing how input features are influencing the output.

# NOTE: After training models like in step 2 of part 3 above cells I should do the below: 

After training your models, you should:
Benchmark all models

Check for overfitting

Visualize prediction errors

✔ Analyze feature importance

✔ Add predictions back into the dataframe

✔ Integrate the best model into Streamlit

✔ (Optional) Perform hyperparameter tuning

✔ (Optional) Use SHAP for interpretability
This is the exact workflow used in professional ML projects.


# Benchmarking: How do we becnhmark for regression models?

For regression we can compare:
- MAE (Mean Absolute Error) 
- RMSE (Root Mean Squared Error) 
- R^2 (Coefficient of +Determination) 
- Overfitting (= Train Error -Test Error) 

In [ ]:
# ---------------------------------------------------------
# MODEL BENCHMARKING (SAFE VERSION)
# ---------------------------------------------------------

import pandas as pd
import altair as alt

alt.data_transformers.enable('json')
alt.theme.enable('opaque')

print("📊 Regression Model Benchmarking:")
print('-' * 50)

# Expected metric names for each model
expected_metrics = {
    "Random Forest": [
        "rf_train_mae", "rf_test_mae",
        "rf_train_rmse", "rf_test_rmse",
        "rf_train_r2", "rf_test_r2"
    ],
    "KNN": [
        "knn_train_mae", "knn_test_mae",
        "knn_train_rmse", "knn_test_rmse",
        "knn_train_r2", "knn_test_r2"
    ],
    "XGBoost": [
        "xgb_train_mae", "xgb_test_mae",
        "xgb_train_rmse", "xgb_test_rmse",
        "xgb_train_r2", "xgb_test_r2"
    ]
}

rows = []

# Loop through each model and collect metrics if they exist
for model_name, metric_vars in expected_metrics.items():
    metrics_exist = all(var in globals() for var in metric_vars)

    if not metrics_exist:
        print(f"⚠️ Warning: Metrics missing for {model_name}. Skipping.")
        continue

    row = {
        "Model": model_name,
        "Train_MAE": globals()[metric_vars[0]],
        "Test_MAE":  globals()[metric_vars[1]],
        "Train_RMSE": globals()[metric_vars[2]],
        "Test_RMSE":  globals()[metric_vars[3]],
        "Train_R2": globals()[metric_vars[4]],
        "Test_R2":  globals()[metric_vars[5]],
    }

    rows.append(row)

# Build DataFrame
if len(rows) == 0:
    raise ValueError("❌ No model metrics found. Train models before running benchmarking.")

model_results = pd.DataFrame(rows)

# Compute overfitting scores
model_results["MAE_Overfit"] = model_results["Train_MAE"] - model_results["Test_MAE"]
model_results["RMSE_Overfit"] = model_results["Train_RMSE"] - model_results["Test_RMSE"]
model_results["R2_Overfit"] = model_results["Train_R2"] - model_results["Test_R2"]

print(model_results.round(4))
print('-' * 50)

In [ ]:
# Model Interpretability: 
# Interpretable ML models are those that provide the variable coefficients that
# directly dictate the relative degree at which it influence the target y values. 


# Feature Importance for Regression Models (Random Forest + XGBoost)
import pandas as pd
import numpy 

print("📊 Computing Feature Importances...")
print('-' * 40)

# Random Forest Feature Importance
rf_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': reg_model.feature_importances_
}).sort_values('importance', ascending=False)

print("🌲 Top 10 Most Important Features (Random Forest):")
print(rf_importance.head(10))
print('-' * 40)

# XGBoost Feature Importance (if trained)
try:
    xgb_importance = pd.DataFrame({
        'feature': all_feature_names,
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("🚀 Top 10 Most Important Features (XGBoost):")
    print(xgb_importance.head(10))
    print('-' * 40)

except NameError:
    print("XGBoost model not trained yet.")

Above results show that sqft is most predictive  in the random forest model 
RF Importance for SQFT is 0.909676 

- Random forest splits on the feature that reduces the variance the most at each node. 
	- Since SQFT has the largest numeric range in the given dataset, and it correlated strongly with price --> SQFT Explains most of the variance by itself.
		
		- Splits? decision-tree based models (Random forest and XGBoost) make predictions by repeatedly splitting the dataset into smaller groups. Each split divides the data into 2 groups that are more homogenous imn price. 
			A feature that creates the biggest reduction in price variance is chosen for the split. 
				- IE) Since the model splits on SQFT, SQFT is the feature that best separates high-pricevs low-price homes, spo the tree uses it to repeatedly divide the data.



- Random Forest tends to "lock onto" one dominant numeric feature.
	- This is a known behavior: Random forest often over emphasizes continuous features because they offer more potential split points.




And high end luxury level is most predictive in the XGBoost model
XGB importance for luxury high end is 0.452563  

- XGBoost sees luxury levels as the strongest predictor 
	- This makes sense since: 
		- A high end luxury level would distinctly separate high price homes from low/mid 
			- And boosting models love strong categorical splits!
				Why? Boosting models build trees sequentially, and each new tree tries to fix the mistakes of the prior ones. Thus a strong categorical feature creates a clean distinction between groups.




- Overall, we ca say sqft is the strongest numeric predictor, and luxury is the strongest categorical predictor. 
	- Bathrooms and bedrooms matter too, but less.


In [ ]:
for col in X_train_scaled.columns:
    if X_train_scaled[col].dtype == 'object':
        print(col)
        print(X_train_scaled[col].head())


In [ ]:
# ============================================
# SHAP INTERPRETABILITY FOR XGBOOST MODEL
# ============================================

import shap
import matplotlib.pyplot as plt

print("🔍 Initializing SHAP Explainer...")
# SHAP expects raw (unscaled) values for tree models, 
# but scaled values work fine for relative importance.
explainer = shap.TreeExplainer(xgb_model)

print("📊 Computing SHAP values (this may take a moment)...")
shap_values = explainer.shap_values(X_train_scaled)

# --------------------------------------------
# GLOBAL FEATURE IMPORTANCE (SUMMARY PLOT)
# --------------------------------------------
print("\n🌎 SHAP Summary Plot (Global Feature Importance)")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train_scaled, plot_type="bar")
plt.show()

# --------------------------------------------
# DETAILED SUMMARY PLOT (BEESWARM)
# --------------------------------------------
print("\n🐝 SHAP Beeswarm Plot (Feature Effects)")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train_scaled)
plt.show()

# --------------------------------------------
# LOCAL EXPLANATION FOR A SINGLE PREDICTION
# --------------------------------------------
sample_index = 0  # change this to inspect different houses
print(f"\n🏠 SHAP Force Plot for Sample #{sample_index}")

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[sample_index, :],
    X_train_scaled.iloc[sample_index, :]
)

In [ ]:
# Top Feature Importance Visualization for Regression Models (Random Forest)
import altair
import pandas as pd

# Select top N features
top_n = 5

# rf_importance must already be created in the feature importance cell
chart_data = rf_importance.head(top_n)

# Create the bar chart
chart = altair.Chart(chart_data).mark_bar().encode(
    x = altair.X('importance:Q', title='Importance'),
    y = altair.Y('feature:N', title='Feature', sort='-x'),
    color = altair.value('#00B8E7'),  # Single color for regression importance
    tooltip = [
        altair.Tooltip('feature:N', title='Feature'),
        altair.Tooltip('importance:Q', title='Importance', format='.4f')
    ]
).configure(
    background='transparent'
).configure_axis(
    labelColor='white',
    titleColor='white'
).configure_title(
    color='white'
).properties(
    title=f'Top {top_n} Feature Importances (Random Forest Regressor)',
    width=600,
    height=400
)

chart

In [ ]:
# Interpreting Random Forest Models (Regression)
# Feature importance analysis with RandomForestRegressor

# Build a dataframe of random forest feature importances
rf_feature_importance = pd.DataFrame({
    'feature': all_feature_names,  # From the scaling/encoding step
    'importance': reg_model.feature_importances_  # From RandomForestRegressor
    # 
}).sort_values('importance', ascending=False) # This is sorted so the most important ferautes appear at the top.

# Print the top 5 most important features
print("✨ Top 5 Most Important Features (Random Forest Regressor):")
print(rf_feature_importance.head(5))

In [ ]:
# Top Feature Importance Visualization (XGBoost Regressor)

top_n = 5

xgb_feature_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

chart_data = xgb_feature_importance.head(top_n)

chart = altair.Chart(chart_data).mark_bar().encode(
    x = altair.X('importance:Q', title='Importance'),
    y = altair.Y('feature:N', title='Feature', sort='-x'),
    color = altair.value('#FFB347'),  # Orange for XGBoost
    tooltip = [
        altair.Tooltip('feature:N', title='Feature'),
        altair.Tooltip('importance:Q', title='Importance', format='.4f')
    ]
).configure(
    background='transparent'
).configure_axis(
    labelColor='white',
    titleColor='white'
).configure_title(
    color='white'
).properties(
    title=f'Top {top_n} Feature Importances (XGBoost Regressor)',
    width=600,
    height=400
)

chart

In [ ]:
# This cell computes permutation importance for KNN Regressor

""" 
This gives us a ranked list oof features that matter ost for KNN 
 
While KNN has no internal importance mechanism, permutation important revelas:
- Which features KNN relies on most
- How much each feature affect prediction error
- Which featutes are irrelvant

The cell uses MAE as the scoring metric -- Whcih makes the importance values directly tied to price preidction 

""" error 


from sklearn.inspection import permutation_importance
import pandas as pd
import numpy

print("🔍 Computing Permutation Importance for KNN...")
print('-' * 40)

# Compute permutation importance on the TEST set (recommended)
knn_perm = permutation_importance(
    knn_model,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='neg_mean_absolute_error'  # MAE is intuitive for price prediction
)

# Build a DataFrame of feature importances
knn_feature_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': knn_perm.importances_mean,
    'std': knn_perm.importances_std
}).sort_values('importance', ascending=False)

print("✨ Top 5 Most Important Features (KNN Permutation Importance):")
print(knn_feature_importance.head(5))

In [ ]:
# Top Feature Importance Visualization (KNN Permutation Importance)

top_n = 5
chart_data = knn_feature_importance.head(top_n)

chart = alt.Chart(chart_data).mark_bar().encode(
    x = alt.X('importance:Q', title='Importance (Increase in MAE)'),
    y = alt.Y('feature:N', title='Feature', sort='-x'),
    color = alt.value('#9B59B6'),  # Purple for KNN
    tooltip = [
        alt.Tooltip('feature:N', title='Feature'),
        alt.Tooltip('importance:Q', title='Importance', format='.4f'),
        alt.Tooltip('std:Q', title='Std Dev', format='.4f')
    ]
).configure(
    background='transparent'
).configure_axis(
    labelColor='white',
    titleColor='white'
).configure_title(
    color='white'
).properties(
    title=f'Top {top_n} Feature Importances (KNN Permutation Importance)',
    width=600,
    height=400
)

chart

# Section 4 - Experiment Tracking:

# Section 4 - Part 1: Data Loading

In [ ]:
# This cell ignores the 3 categories of warnings shown below: 

import warnings

# Filter out ResourceWarnings. These typically occur when libraries open temporary files 
# or network connections, and they are being ignored since they are harmless in ML notebooks. 
warnings.filterwarnings('ignore', category = ResourceWarning)

# Filter out DeprecationWarning. These typically tell you a function or parameter will be removed in 
# a future version. These are ignored as they are unneeded information during analysis reports.
warnings.filterwarnings('ignore', category = DeprecationWarning)

# Filter out UserWarning are general warnings triggered by libraries when something might be unexpected but not harmful. 
warnings.filterwarnings('ignore', category = UserWarning)

In [ ]:
# This cell loads the dataset from the combined sql table and displays it 
# with streamlit.

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Sesh.table uses the snowflake session to convert the table named HOUSE_DATA_FINAL into a 
# pandas dataframe.
house_data_df3 = sesh.table('HOUSE_DATA_FINAL').to_pandas()

# Below prints a title and the dimensions of the bear_df dataframe.
streamlit.write("📊 House Dataset Loaded:")
streamlit.write(f"Shape: {house_data_df3.shape}")

house_data_df3 # Displays the bear_df dataframe.

# Section 4 - Part 2: Data Preprocessing

In [ ]:
house_data_df3.columns # Gives a Pandas Index object of the names of all the columns in the dataframe.

In [ ]:
# Prepare features and target
# Below is used to split the dataset into training and test data sets.
from sklearn.model_selection import train_test_split

# X contains all the input features we use to predict Y (Price).  
# This forms a list of features (feature matrix)
X = house_data_df3.drop(columns = ['price', 'house_id', 'address', 'url_link', 'image_path'])

# Below selects the column to target. 
# This is what the model will learn to classify.
y = house_data_df3['price']

In [ ]:
# Check for missing data

# Below counts the missing values in the feature matrix.
missing_features = X.isnull().sum().sum()

# Below coutns the missing values in the target y.
missing_target = y.isnull().sum()

# Below adds a subheader in the Streamlit UI.
streamlit.subheader("🔍 Data Quality Check:")

# Below prints the number of missing feature values.
streamlit.write(f"Missing feature values: `{missing_features}`")

# Below prints the number of missing target values.
streamlit.write(f"Missing target values: `{missing_target}`")

In [ ]:
# Below splits the dataset into training and testing sets:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, # x is the features and y is the target labels 
    test_size = 0.2, # Sets the test set to be 20% of the data 
    random_state = 42, # Random_state is a seed for the random number generator.
    # No stratify for regression
)

streamlit.write("✅ Data preparation completed!")

# Below displays the spliut summary, and this shows:
# Number of training samples, number of testing samples, percentage split, number of features (Columns)
streamlit.subheader("📊 Data Split Summary:")
streamlit.write(f"Training set: `{X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)`")
streamlit.write(f"Testing set: `{X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)`")
streamlit.write(f"Number of features: {X_train.shape[1]}")

# Below shows the class distribution in each split:
# This is done to ensure stratification worked, confirms no class was lost during splitting,
# helps detect imbalance issues early. 
streamlit.subheader("🎯 Target Summary:")
streamlit.write(f"Training target stats: `{y_train.describe().to_dict()}`")
streamlit.write(f"Testing target stats: `{y_test.describe().to_dict()}`")

In [ ]:
# We perform the following preprocessing: 
# 1) StandardScaler for numerical features. This transforms features to have mean=0 
# and std=1 
# 2) OneHotEncoder for categorical features (luxury lelve, house type, lot size)
# This converts categorical variables into binary columns. 

# Feature scaling and preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify numerical and categorical columns by looking at the data types in X_train
# Selects int64 and float64 as the numerical features and object s as the categorical features.
numerical_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object']).columns

# Below scales the numerical features
# StandardScaler scales the features so mean equals zero and standard deviation is equal to
# This is done to prevent large scale features (big values like weight) from dominating 
# small scale ones (Like age). 
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numerical_features])
X_test_scaled_num = scaler.transform(X_test[numerical_features])

# One-Hot encode categorical features.
# Below handles categorical features by converting each into a binary column.
# We also ensure to ignore any categories not seen inthe test set.
onehot = OneHotEncoder(sparse_output = False, handle_unknown='ignore')
X_train_scaled_cat = onehot.fit_transform(X_train[categorical_features])
X_test_scaled_cat = onehot.transform(X_test[categorical_features])

# Get categorical feature names after one-hot encoding and replace spaces with underscores
cat_feature_names = [] # Holds the names of the categorical features.
# Below creates feature names that are to be added to the above list.
# zip() takes 2 or more lists and pairs their ele
for feature, categories in zip(categorical_features, onehot.categories_):
    for category in categories:
        cat_feature_names.append(f"{feature}_{category}".replace(' ', '_').lower())

# Below combines numerical and categorical features matrices into a single one.
X_train_scaled = numpy.hstack([X_train_scaled_num, X_train_scaled_cat])
X_test_scaled = numpy.hstack([X_test_scaled_num, X_test_scaled_cat])

# Below converts a list of all the feature names to a DataFrame with proper column names
# This is done for easier dubgging, easier feature importance analysis, easier visuialization,
# and it preserves row alignment with the original dataset.  
all_feature_names = list(numerical_features) + cat_feature_names
X_train_scaled = panda.DataFrame(X_train_scaled, columns = all_feature_names, index = X_train.index)
X_test_scaled = panda.DataFrame(X_test_scaled, columns = all_feature_names, index = X_test.index)


# Below displays the results using Streamlit:
streamlit.write("Feature scaling completed!")

streamlit.subheader("Feature Processing:")
streamlit.write(f"Numerical features: `{numerical_features.tolist()}`")
streamlit.write(f"Categorical features: `{categorical_features.tolist()}`")

streamlit.subheader("Scaled Data Dimensions:")
streamlit.write(f"Training features: `{X_train_scaled.shape}`")
streamlit.write(f"Testing features: `{X_test_scaled.shape}`")

# Display first few encoded feature names
streamlit.write("\nFirst few encoded feature names:")
streamlit.write(all_feature_names[:10])

In [ ]:
y_train_encoded

In [ ]:
y_test_encoded

In [ ]:
# Save scaled test data + actual price values to Snowflake

# Create a copy of the scaled test features
test_df = X_test_scaled.copy()

# Add the actual price values (regression target)
test_df['ACTUAL_PRICE'] = y_test

# Convert pandas DataFrame to Snowpark DataFrame
snowpark_df = sesh.create_dataframe(test_df)

# Write to Snowflake table
snowpark_df.write.mode("overwrite").save_as_table(
    "SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_TEST_DATA"
)

# Display results
streamlit.write("✅ Data successfully saved to HOUSE_TEST_DATA table!")
streamlit.write(f"Number of rows: {len(test_df)}")
streamlit.write(f"Number of columns: {len(test_df.columns)}")

In [ ]:
# Outlier Detection & Removal using IQR

streamlit.subheader("🧹 Outlier Detection & Removal")

# Below find the 25th percentile of price (25% homes cost less than this value)
Q1 = y.quantile(0.25)

# Below find the 75th percentile (75% of homes cost less than this value)
Q3 = y.quantile(0.75)

# Below computes the interquartile range (Measures the "middle spread" of the data 
# IQR is a measure of variabiliity, and is not affected by extreme values. 
# Its not effected by extremem values as it only looks at ahe middle 50% of the data. thus extreme high and low values ar eignored.
IQR = Q3 - Q1

# Below defines the lowest aceeptable price 
# Anythong below is considered an outlier. 
# The 1.5xIQR rule is standard statistical method.
lower_bound = Q1 - 1.5 * IQR

# Below defines the highest acceptable price
# Anythign above is an outlier.
upper_bound = Q3 + 1.5 * IQR

streamlit.write(f"Lower bound: `{lower_bound:.2f}`")
streamlit.write(f"Upper bound: `{upper_bound:.2f}`")

# Filter dataset to keep only data in range of lower to upper bound. 
before_rows = len(house_data_df3)
house_data_df3 = house_data_df3[(house_data_df3['price'] >= lower_bound) &
                                (house_data_df3['price'] <= upper_bound)]
after_rows = len(house_data_df3)

streamlit.write(f"Removed `{before_rows - after_rows}` outlier rows.")
streamlit.write(f"Remaining rows: `{after_rows}`")

# Outliers distort scaling, inflat e error metrics, and confuse KNN my modes.

In [ ]:
# Outlier Detection & Removal using IQR

streamlit.subheader("🧹 Outlier Detection & Removal")

# Below find the 25th percentile of price (25% homes cost less than this value)
Q1 = y.quantile(0.25)

# Below find the 75th percentile (75% of homes cost less than this value)
Q3 = y.quantile(0.75)

# Below computes the interquartile range (Measures the "middle spread" of the data 
# This is a measure of variabiliity (is not affected by extreme values? ))
IQR = Q3 - Q1

# Below defines the lowest aceeptable price 
# Anythong below is considered an outlier. 
# The 1.5xIQR rule is standard statistical method.
lower_bound = Q1 - 1.5 * IQR

# Below defines the highest acceptable price
# Anythign abov
upper_bound = Q3 + 1.5 * IQR

streamlit.write(f"Lower bound: `{lower_bound:.2f}`")
streamlit.write(f"Upper bound: `{upper_bound:.2f}`")

# Filter out outliers
before_rows = len(house_data_df3)
house_data_df3 = house_data_df3[(house_data_df3['price'] >= lower_bound) &
                                (house_data_df3['price'] <= upper_bound)]
after_rows = len(house_data_df3)

streamlit.write(f"Removed `{before_rows - after_rows}` outlier rows.")
streamlit.write(f"Remaining rows: `{after_rows}`")

# Outliers distort scaling, inflat e error metrics, and confuse KNN my modes.

In [ ]:
# Rare categoriy handling: 

# If a categorical feature has categories 
# that appear only 1–2 times, one‑hot encoding will create:
# Unstable columns, noise and overfitting risk. 
# So by grouping these rare categories into other, we can prevent his.

# Rare Category Handling for Categorical Features

streamlit.subheader("🔎 Rare Category Handling")

threshold = 10  # categories with <10 occurrences will be grouped into "Other"

for col in house_data_df3.select_dtypes(include=['object']).columns:
    value_counts = house_data_df3[col].value_counts()
    rare_categories = value_counts[value_counts < threshold].index

    if len(rare_categories) > 0:
        house_data_df3[col] = house_data_df3[col].replace(rare_categories, "Other")
        streamlit.write(f"Column `{col}`: grouped {len(rare_categories)} rare categories into 'Other'")

In [ ]:
# Feature engineering: 

streamlit.subheader("🛠️ Feature Engineering")

# Example engineered features (customize based on your dataset)
if 'sqft' in house_data_df3.columns and 'price' in house_data_df3.columns:
    house_data_df3['price_per_sqft'] = house_data_df3['price'] / house_data_df3['sqft']

if 'year_built' in house_data_df3.columns:
    house_data_df3['house_age'] = 2025 - house_data_df3['year_built']

if 'bedrooms' in house_data_df3.columns and 'sqft' in house_data_df3.columns:
    house_data_df3['bedrooms_per_sqft'] = house_data_df3['bedrooms'] / house_data_df3['sqft']

if 'bathrooms' in house_data_df3.columns and 'bedrooms' in house_data_df3.columns:
    house_data_df3['bath_bed_ratio'] = house_data_df3['bathrooms'] / house_data_df3['bedrooms']

streamlit.write("New engineered features added:")
streamlit.write([col for col in house_data_df3.columns if col in 
                 ['price_per_sqft', 'house_age', 'bedrooms_per_sqft', 'bath_bed_ratio']])


Other possible engineered feature we may want to use in a home price prediction project:

🏠 Structural Features
- total_rooms = bedrooms + bathrooms
- rooms_per_sqft = total_rooms / sqft
- bathroom_to_bedroom_ratio = bathrooms / bedrooms
- finished_sqft_ratio = finished_sqft / total_sqft

📍 Location Features
If you have coordinates or neighbourhood:
- is_urban = 1 if neighbourhood in urban_list else 0
- distance_to_city_center
- school_quality_score
- crime_rate_index

🕒 Time‑Based Features
If you have year built or renovation year:
- house_age = current_year - year_built
- years_since_renovation = current_year - renovation_year

💰 Price‑Related Features
- price_per_sqft = price / sqft
- lot_price_ratio = price / lot_size
- luxury_flag = 1 if price > threshold else 0

🌳 Lot / Exterior Features
- lot_size_per_sqft = lot_size / sqft
- garage_space_ratio = garage_spaces / bedrooms

🧹 Binary Flags
- has_garage
- has_basement
- has_fireplace
- is_renovated

# Section 4 - Part 3: Experiment Tracking Setup

In [ ]:
# We do experiment tracking using snowflakes ML experiemnt tracking 
# This allows us to log and compare different models and their hyperparameters.

# Below imports snowflake's experiment tracking libary. This lets us log models, metrics 
# hyperparameters, compare runs, store artifacts, and keep a history of experiments.
from snowflake.ml.experiment.experiment_tracking import ExperimentTracking

# Create an ExperimentTracking object that is tracking the current snowflake session.
exp = ExperimentTracking(session = sesh)

# Below sets the experiment name
experiment_name = "House_Price_Regression_Experiment"
# Below ensures snowflake logs all future model runs. 
exp.set_experiment(experiment_name)

# Below confirms the ExperimentTracking is initialized.
st.write(f"✅ Experiment Tracking Initialized: `{experiment_name}`")

In [ ]:
# This cell does the following:
# Trains the random forest model, evaluates the model, logs hyperparameters to snowflake,
# logs metrics, and stores the results inside of the ExperimentTracker

# Below imports what we need for the model and its evaluation.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime
import numpyu

# Define Hyperparameters. 
# These are the settings that control how the random forest behaves.
params = {
    # Below builds 200 decision trees.
    # More trees --> more stable predictions
    # Too many trees? --> slow training with diminishing returns.
    "n_estimators": 200,
    
    # Below ensures trees can only be 10 levels deep (prevents overfitting)
    # unless we have a massive dataset with diverse data points, we should tress with lower depth. 
    # Deep trees memorize the training data --> leads to overfitting
    # DShallow treees generalize better.    
    "max_depth":10,

    
    # Below ensures each leaf must have at least 2 samples.
    # Each leaf must contain at least 2 samples: 
    # This prevents leaves with 1 sample (overftting) and smooth predictions.
    "min_samples_leaf": 2,
    
    # Below ensures each split considers a sqrt(num_features) number of random features.  
    # At each split, the model consiedrs only sqrt(num_features) random features.
    # Adds randomness, reduces correlation between trees, and improves generalization
    'max_features': 'sqrt',
    
    # Below ensures reproducibility.
    # Same data --> same model --> Same results 
    # Essential for experiment tracking
    "random_state": 42
}

# Below creates a unique run name with the timestamp. This ensures that every 
# experiments get a unique ID, and thus makes it easier to compare runs later one.
run_name = f"rf_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Below starts an experiment run:
# Everything inside the with block is contained to this experiment run.
# Inside this block we log, hyperparameters, metrics, and model performance.
with exp.start_run(run_name):
    # Below logs hyperparameters. IE) Stores the models' setting inside snowflake.
    exp.log_params(params)

    # Below trains the model by fiurst rceating a random firest classifier using the 
    # hyperparameters. Then .fit() trains it on the scaled training data.  
    model = RandomForestRegressor(**params)
    model.fit(X_train_scaled, y_train)

    # Below makes a prediction on the test set:
    y_pred = model.predict(X_test_scaled)

    # Below computes metrics for evaluation:

    # MAE measures the avg absolute difference between predicted and actual price
    mae = mean_absolute_error(y_test, y_pred)

    # RMSE measures the square root of the average squared error.
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # R^2  is the coefficient of determination and it measure how much 
    # of the variation in rice that the model explains.
    r2 = r2_score(y_test, y_pred)
    

    # Below logs the metric by storing the model's performance inside the experiment tracker.
    exp.log_metric("MAE", mae)
    exp.log_metric("RMSE", rmse)
    exp.log_metric("R2", r2)


    # Below displays the results with streamlit
    streamlit.write("📊 Model Performance (Random Forest Regressor):")
    streamlit.write(f"- MAE: `{mae:.4f}`")
    streamlit.write(f"- RMSE: `{rmse:.4f}`")
    streamlit.write(f"- R² Score: `{r2:.4f}`")

# Section 4 - Part 4: Hyperparameter Tuning:

In [ ]:
# Now we are going to analyze the results from our 
# random Forest hyperparameter tuning epxeriments. 

# Below imports all that is needed:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, matthews_corrcoef
from datetime import datetime # Used to generate unique run names.
import itertools # Used to generate all hyperparameter combinations.

# Below defines the hyperparameter grid, which is the set of value you want to try for each hyperparameter.
param_grid = {
    # Below says we try 100 and 200 trees.
    "n_estimators": [100, 200],
    # Below says we try depth of 3 and depth of 5.
    "max_depth": [3, 5],
    # Below says we try leaf sizes of 1,2 and 5.
    "min_samples_leaf": [1, 2, 5],
    # Below says we try feature selection strategies "sqrt" and "log2"
    "max_features": ['sqrt', 'log2']
}

# Below initializes the list to store results
results = []

# Below generates all combinations of parameters
# itertools.product creates every possible combiantion. 
# zip pairs parameter names with values. 
# dict() turns each pair into a dictionary.
# In total we train 24 different random forest models. We train so many as we dont know 
# ahead of time which combiantion of hyperparameters will perform the best. 
param_combinations = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

# Below loops through every possible parameter combiantoin:
for params in param_combinations:
    # Add a fixed random state to params for reproducibility across runs.
    params['random_state'] = 42
    
    # Below creates a unique run name with timestamp and params summary
    run_name = f"RF_tune_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Below creates an experiment run, and everything inside of this block gets logged to Snowflake.
    # This includes: Hyperparameters, metrics, run name.
    with exp.start_run(run_name):
        # Below logs the hyperparameters that are used in this run.
        exp.log_params(params)

        # Below trains a new random forest model with the current hyperparamters.
        model = RandomForestClassifier(**params)
        model.fit(X_train_scaled, y_train_encoded)

        # Below pedicts the test set.
        y_pred = model.predict(X_test_scaled)

        # Below calculates the evaluation metrics
        acc = accuracy_score(y_test_encoded, y_pred) # Overall correctness
        precision = precision_score(y_test_encoded, y_pred, average = 'macro') # How pure predictions are across classes 
        recall = recall_score(y_test_encoded, y_pred, average = 'macro') # How well each class is recovered.
        mcc = matthews_corrcoef(y_test_encoded, y_pred) 

        # Log metrics to snowflake
        exp.log_metric("accuracy", acc)
        exp.log_metric("precision", precision)
        exp.log_metric("recall", recall)
        exp.log_metric("mcc", mcc)

        # Store results in snowflake by creating a python list of dictionaries. 
        # This is done so we can convert to a dataframe, sort by accuracy, visualize results.
        results.append({
            'run_name': run_name,
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_samples_leaf': params['min_samples_leaf'],
            'max_features': params['max_features'],
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'mcc': mcc
        })

        streamlit.write(f"Parameters: {params}")

In [ ]:
# With hyperparameter tuning finished, we can analyze the results of the 24 models 
# to find the best performing one.

# Below creates a DataFrame from the results list by converting it into a dataframe. 
results_df = pd.DataFrame(results)

# Below displays the summary statistics table in streamlit.
# This ensures the entire tuning table is displayed, the best value in each metric column is highlighted in green, 
# and this is done since it is easy to visually scan for top performers. 
streamlit.write("📊 Model Performance Summary:")
streamlit.dataframe(results_df.style.highlight_max(subset = ['accuracy', 'precision', 'recall', 'mcc'], color = "green"))

# Below displays the best performing configuration
# idxmax() finds the row index with thehighest accruacy.
# .loc[] retrives that entire row
# Thus best_model is a single row containing the information on:
# the best hyperparameters, the best accruacy, the best precision, the best recall, 
# the best MCC, the run name.
best_model = results_df.loc[results_df['accuracy'].idxmax()]


# Below displays the best model's performance.
streamlit.subheader("🏆 Best Model Configuration:")
streamlit.write("Learning Algorithm: `Random Forest`")
streamlit.write(f"Accuracy: `{best_model['accuracy']:.4f}`")
streamlit.write(f"Precision: `{best_model['precision']:.4f}`")
streamlit.write(f"Recall: `{best_model['recall']:.4f}`")
streamlit.write(f"MCC: `{best_model['mcc']:.4f}`")
streamlit.write(f"Learning Parameters:")
streamlit.write(f"- n_estimators: `{best_model['n_estimators']}`")
streamlit.write(f"- max_depth: `{best_model['max_depth']}`")
streamlit.write(f"- min_samples_leaf: `{best_model['min_samples_leaf']}`")
streamlit.write(f"- max_features: `{best_model['max_features']}`")

# Section 4 - Part 5: Model Registry

In [ ]:
# Now we register the best model to Snowflake's Model 
# Registry for deployment and management 

# Train the final RandomForestRegressor using the best hyperparameters

# Extract best hyperparameters from your tuning results
best_params = {
    'n_estimators': int(best_model['n_estimators']),
    'max_depth': int(best_model['max_depth']),
    'min_samples_leaf': int(best_model['min_samples_leaf']),
    'max_features': best_model['max_features'],
    'random_state': 42
}

# Train the final regression model
final_model = RandomForestRegressor(**best_params)
final_model.fit(X_train_scaled, y_train)

streamlit.write("✅ Final regression model trained with best hyperparameters.") 

In [ ]:
best_params # Displays the best parameters object.

In [ ]:
# Register the best-performing regression model in Snowflake Model Registry
# This cell:
# Registers the model
# logs regression metrics
# Stores sample input 
# creates a versioned model entry

from snowflake.ml.registry import Registry
from datetime import datetime

# Temporarily suppress warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Connect to the Model Registry
registry = Registry(sesh)

# Clean column names for Snowflake compatibility
X_train_clean = X_train_scaled.copy()
X_train_clean.columns = X_train_clean.columns.str.replace(' ', '_')

# Define model name and version
model_name = "HOUSE_PRICE_REGRESSOR"
model_version = f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Register the model in snowflake
model_ref = registry.log_model(
    model=final_model, # Registers the model

    # Creates a versioned model entry
    model_name=model_name,
    version_name=model_version,
    sample_input_data=X_train_clean.head(5), # Stores sample input
    metrics={ # Logs regression metrics
        'MAE': float(best_model['MAE']),
        'RMSE': float(best_model['RMSE']),
        'R2': float(best_model['R2'])
    },
    options={
        "case_sensitive": True
    },
    comment="Best performing Random Forest Regressor from hyperparameter tuning"
)

# Reset warnings
warnings.resetwarnings()

streamlit.write("✅ Regression model successfully registered!")
streamlit.write(f"Model Name: `{model_name}`")
streamlit.write(f"Version: `{model_version}`")

In [ ]:
// Below shows the available versions in a model
SHOW VERSIONS IN MODEL house_type_classifier;

In [ ]:
// Below shows the available functions in a model
SHOW FUNCTIONS IN MODEL house_type_classifier

In [ ]:
# Deploy the regression model as a Snowflake service

# What is a snowflake service? 
# A fully managed, scalable endpoint inside Snowflake that hosts your ML model so it can be called by applications, APIs, dashboards, or Streamlit apps.
# Above defintion taken from Microsoftr Copilot.

# This cell: 
# Drops old services 
# Deploys the regression model 
# makes it callable from Streeamlit or APIS


# Drop existing service if it exists
sesh.sql("DROP SERVICE IF EXISTS house_price_regressor").collect()

# Deploy the new service
model_ref.create_service(
    service_name="house_price_regressor",
    service_compute_pool="system_compute_pool_cpu",
    ingress_enabled=True,
    gpu_requests=None
)

streamlit.write("✅ House Price Regression model service created successfully!")

In [ ]:
// Show service endpoints exposed by the mode 

// WHAT DOES IT MEAN TO BE EXPOSED BY THE MODEL?? ask AI todo

SHOW SERVICES;

In [ ]:
SELECT HOUSE_RF_CLASSIFIER

In [ ]:
SHOW ENDPOINTS IN SERVICE HOUSE_RF_CLASSIFIER;

In [ ]:
SHOW TABLES;

# NOTE BELOW 2 CELLS WONT WORK --> NEED TO BE FIXED.


# Ask AI how these 2 can be changed to the curretn project.

In [ ]:
-- With the model deployed as a setvice, wre can use it to make predictions on new data
-- We do this by:
-- Query the HOUSE_TEST_DATA table
-- Pass features to the model

/* 
HOUSE_RF_CLASSIFIER 
    The name of the deployed model 

! PREDICT() 
    Snowflake syntax -- Says to call the PREDICT function on that model.

Read like "Use HOUSE_RF_CLASSIFIER to predict given these features."
*/

SELECT
  HOUSE_RF_CLASSIFIER ! PREDICT(
    "price",
    "bedrooms",
    "bathrooms",
    "sqft",
    "luxury_basic",
    "luxury_mid",
    "luxury_highend",
    "lotsize_small", 
    "lotsize_medium",
    "lotsize_large",
    "house_type_modern",
    "house_type_ranch",
    "house_type_raised_ranch",
    "house_type_two_story_trad",
    "house_type_bungalow",
    "house_type_prairie_suburban",
    "house_type_farmhouse", 
    "house_type_chalet", 
    "house_type_four_square",
    "house_type_skinny", 
    "house_type_mediterranean"
  ) AS predicted_price
FROM
  SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_TEST_DATA
LIMIT
  5;

In [ ]:
-- Average values of features
SELECT
  AVG("body_mass_kg") AS "body_mass_kg",
  AVG("shoulder_hump_height_cm") AS "shoulder_hump_height_cm",
  AVG("claw_length_cm") AS "claw_length_cm",
  AVG("snout_length_cm") AS "snout_length_cm",
  AVG("forearm_circumference_cm") AS "forearm_circumference_cm",
  AVG("ear_length_cm") AS "ear_length_cm",

  -- Fur Color Proportions
  AVG(CASE WHEN "fur_color" = 'Black' THEN 1 ELSE 0 END) AS "fur_color_black",
  AVG(CASE WHEN "fur_color" = 'Blackish-Brown' THEN 1 ELSE 0 END) AS "fur_color_blackish_brown",
  AVG(CASE WHEN "fur_color" = 'Blond' THEN 1 ELSE 0 END) AS "fur_color_blond",
  AVG(CASE WHEN "fur_color" = 'Brown' THEN 1 ELSE 0 END) AS "fur_color_brown",
  AVG(CASE WHEN "fur_color" = 'Cinnamon' THEN 1 ELSE 0 END) AS "fur_color_cinnamon",
  AVG(CASE WHEN "fur_color" = 'Dark Brown' THEN 1 ELSE 0 END) AS "fur_color_dark_brown",
  AVG(CASE WHEN "fur_color" = 'Grizzled' THEN 1 ELSE 0 END) AS "fur_color_grizzled",
  AVG(CASE WHEN "fur_color" = 'Light Brown' THEN 1 ELSE 0 END) AS "fur_color_light_brown",
  AVG(CASE WHEN "fur_color" = 'Medium Brown' THEN 1 ELSE 0 END) AS "fur_color_medium_brown",
  AVG(CASE WHEN "fur_color" = 'Reddish-Brown' THEN 1 ELSE 0 END) AS "fur_color_reddish_brown",

  -- Facial Profile Proportions
  AVG(CASE WHEN "facial_profile" = 'Dished' THEN 1 ELSE 0 END) AS "facial_profile_dished",
  AVG(CASE WHEN "facial_profile" = 'Straight' THEN 1 ELSE 0 END) AS "facial_profile_straight",

  -- Paw Pad Texture Proportions
  AVG(CASE WHEN "paw_pad_texture" = 'Rough' THEN 1 ELSE 0 END) AS "paw_pad_texture_rough",
  AVG(CASE WHEN "paw_pad_texture" = 'Smooth' THEN 1 ELSE 0 END) AS "paw_pad_texture_smooth"

FROM SNOWFLAKE_LEARNING_DB.PUBLIC.BEAR_TEST_DATA
WHERE "species" ILIKE '%American Black Bear%';